# From 8 minutes to 4 seconds
## Solving large systems with Jacobi + GMRES

**Romain Sacchi (PSI) · Brightcon 2026 · 25 September 2026 · 8 minutes**<br>
Open Tools and Development · Aalborg University & online

> Direct sparse factorisation is an excellent default—until fill-in makes memory and runtime scale much faster than the matrix itself. An iterative solve makes that trade-off explicit and controllable.


## Run of show

1. Show how size and density create direct-solver fill-in.
2. Use a large banded counterexample to show that size alone is not decisive.
3. Reuse one fixed matrix and watch factorization reuse reverse the winner.
4. Rebuild paired matrices across 500 Monte Carlo iterations, with and without warm starts.
5. Translate the evidence back to `bw2calc`.

Every benchmark runs live in an isolated worker. Unsafe cases are marked **SKIPPED**, and failures
or timeouts remain visible instead of silently disappearing. The grids are calibrated to run
top-to-bottom in under six minutes in the reference `jacobi` environment.


In [ ]:
# Notebook helpers
# ----------------
# These two functions create a sparse technosphere-like matrix and solve Ax = b.
# The benchmark cells below use them directly; the larger worker code is kept private.

import time
from pathlib import Path
import numpy as np
from scipy import sparse
from scipy.sparse.linalg import LinearOperator, gmres, spsolve
# Build a matrix with a fixed number of links per activity.
def constant_degree_matrix(
    n: int, degree: int, seed: int, diagonal_span: float
) -> sparse.csc_matrix:
    if degree < 1 or degree >= n:
        raise ValueError("degree must be between 1 and n - 1")
    rng = np.random.default_rng(seed)
    columns = np.repeat(np.arange(n, dtype=np.int64), degree)
    rows = rng.integers(0, n, size=n * degree, dtype=np.int64)
    diagonal_hits = rows == columns
    rows[diagonal_hits] = (rows[diagonal_hits] + 1) % n

    # Keep each column's absolute off-diagonal sum comfortably below one.
    coefficients = rng.uniform(0.01, 0.04, size=n * degree)
    off_diagonal = sparse.coo_matrix(
        (-coefficients, (rows, columns)), shape=(n, n)
    ).tocsc()
    matrix = sparse.eye(n, format="csc") + off_diagonal
    row_scales = 10 ** rng.uniform(-diagonal_span / 2, diagonal_span / 2, size=n)
    matrix = sparse.diags(row_scales, format="csc") @ matrix
    matrix.sum_duplicates()
    matrix.eliminate_zeros()
    matrix.sort_indices()
    return matrix

# Build a locally connected matrix with low direct-solver fill-in.
def banded_matrix(
    n: int,
    half_bandwidth: int,
    seed: int,
    diagonal_span: float,
    coupling: float = 0.8,
) -> sparse.csc_matrix:
    """Build a sparse, locally connected matrix with predictable low LU fill-in."""
    if half_bandwidth < 1 or half_bandwidth >= n:
        raise ValueError("half_bandwidth must be between 1 and n - 1")
    if not 0 < coupling < 1:
        raise ValueError("coupling must be between zero and one")

    coefficient = coupling / (2 * half_bandwidth)
    offsets = [
        offset for offset in range(-half_bandwidth, half_bandwidth + 1) if offset
    ]
    diagonals = [
        -coefficient * np.ones(n - abs(offset), dtype=float) for offset in offsets
    ]
    matrix = sparse.eye(n, format="csc") + sparse.diags(
        diagonals, offsets, shape=(n, n), format="csc"
    )

    rng = np.random.default_rng(seed)
    row_scales = 10 ** rng.uniform(-diagonal_span / 2, diagonal_span / 2, size=n)
    matrix = (sparse.diags(row_scales, format="csc") @ matrix).tocsc()
    matrix.sort_indices()
    return matrix

# Solve with a direct method or with GMRES, optionally scaled by the diagonal.
def solve(
    matrix: sparse.csc_matrix,
    demand: np.ndarray,
    solver: str,
    rtol: float,
    restart: int,
    maxiter: int,
    x0: np.ndarray | None = None,
) -> tuple[np.ndarray, int | None, int]:
    if solver == "numpy-dense":
        return np.linalg.solve(matrix.toarray(), demand), None, 0
    if solver == "superlu":
        return spsolve(matrix, demand, use_umfpack=False), None, 0
    if solver == "umfpack":
        import scikits.umfpack  # noqa: F401

        return spsolve(matrix, demand, use_umfpack=True), None, 0
    if solver == "pardiso":
        from pypardiso import spsolve as pardiso_spsolve

        return pardiso_spsolve(matrix, demand), None, 0

    residual_history: list[float] = []
    preconditioner = None
    if solver == "jacobi-gmres":
        diagonal = matrix.diagonal()
        if np.any(diagonal == 0):
            raise ValueError("Jacobi requires a non-zero matrix diagonal")
        inverse_diagonal = 1.0 / diagonal
        preconditioner = LinearOperator(
            matrix.shape,
            matvec=lambda vector: inverse_diagonal * vector,
            dtype=matrix.dtype,
        )
    elif solver != "gmres":
        raise ValueError(f"Unknown solver: {solver}")

    solution, info = gmres(
        matrix,
        demand,
        x0=x0,
        M=preconditioner,
        rtol=rtol,
        atol=0.0,
        restart=restart,
        maxiter=maxiter,
        callback=residual_history.append,
        callback_type="pr_norm",
    )
    return solution, int(info), len(residual_history)


import tempfile
import json
import subprocess
import sys

_embedded_dir = Path(tempfile.mkdtemp(prefix="jacobi_notebook_"))
_embedded_worker = _embedded_dir / "benchmark_synthetic.py"
_embedded_suite = _embedded_dir / "run_synthetic_suite.py"
_embedded_worker.write_text('"""Run one isolated synthetic linear-system benchmark and emit JSON.\n\nEach invocation handles one matrix and one solver so peak RSS measurements are\nnot contaminated by allocations retained by an earlier solver.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport threading\nimport time\nfrom dataclasses import asdict, dataclass\nfrom typing import Callable\n\nimport numpy as np\nimport psutil\nimport scipy\nfrom scipy import sparse\nfrom scipy.sparse.linalg import LinearOperator, gmres, splu, spsolve\n\n\n@dataclass\nclass MemoryMonitor:\n    interval_seconds: float = 0.002\n\n    def __post_init__(self) -> None:\n        self.process = psutil.Process()\n        self.baseline_bytes = self.process.memory_info().rss\n        self.peak_bytes = self.baseline_bytes\n        self._stop = threading.Event()\n        self._thread = threading.Thread(target=self._poll, daemon=True)\n\n    def _poll(self) -> None:\n        while not self._stop.is_set():\n            self.peak_bytes = max(self.peak_bytes, self.process.memory_info().rss)\n            time.sleep(self.interval_seconds)\n\n    def __enter__(self) -> "MemoryMonitor":\n        self._thread.start()\n        return self\n\n    def __exit__(self, *_: object) -> None:\n        self._stop.set()\n        self._thread.join()\n        self.peak_bytes = max(self.peak_bytes, self.process.memory_info().rss)\n\n    @property\n    def incremental_peak_bytes(self) -> int:\n        return max(0, self.peak_bytes - self.baseline_bytes)\n\n\ndef constant_degree_matrix(\n    n: int, degree: int, seed: int, diagonal_span: float\n) -> sparse.csc_matrix:\n    if degree < 1 or degree >= n:\n        raise ValueError("degree must be between 1 and n - 1")\n    rng = np.random.default_rng(seed)\n    columns = np.repeat(np.arange(n, dtype=np.int64), degree)\n    rows = rng.integers(0, n, size=n * degree, dtype=np.int64)\n    diagonal_hits = rows == columns\n    rows[diagonal_hits] = (rows[diagonal_hits] + 1) % n\n\n    # Keep each column\'s absolute off-diagonal sum comfortably below one.\n    coefficients = rng.uniform(0.01, 0.04, size=n * degree)\n    off_diagonal = sparse.coo_matrix(\n        (-coefficients, (rows, columns)), shape=(n, n)\n    ).tocsc()\n    matrix = sparse.eye(n, format="csc") + off_diagonal\n    row_scales = 10 ** rng.uniform(-diagonal_span / 2, diagonal_span / 2, size=n)\n    matrix = sparse.diags(row_scales, format="csc") @ matrix\n    matrix.sum_duplicates()\n    matrix.eliminate_zeros()\n    matrix.sort_indices()\n    return matrix\n\n\ndef banded_matrix(\n    n: int,\n    half_bandwidth: int,\n    seed: int,\n    diagonal_span: float,\n    coupling: float = 0.8,\n) -> sparse.csc_matrix:\n    """Build a sparse, locally connected matrix with predictable low LU fill-in."""\n    if half_bandwidth < 1 or half_bandwidth >= n:\n        raise ValueError("half_bandwidth must be between 1 and n - 1")\n    if not 0 < coupling < 1:\n        raise ValueError("coupling must be between zero and one")\n\n    coefficient = coupling / (2 * half_bandwidth)\n    offsets = [\n        offset for offset in range(-half_bandwidth, half_bandwidth + 1) if offset\n    ]\n    diagonals = [\n        -coefficient * np.ones(n - abs(offset), dtype=float) for offset in offsets\n    ]\n    matrix = sparse.eye(n, format="csc") + sparse.diags(\n        diagonals, offsets, shape=(n, n), format="csc"\n    )\n\n    rng = np.random.default_rng(seed)\n    row_scales = 10 ** rng.uniform(-diagonal_span / 2, diagonal_span / 2, size=n)\n    matrix = (sparse.diags(row_scales, format="csc") @ matrix).tocsc()\n    matrix.sort_indices()\n    return matrix\n\n\ndef fixed_density_matrix(\n    n: int,\n    density: float,\n    seed: int,\n    diagonal_span: float,\n    family: str = "lca-random",\n    blocks: int = 8,\n) -> sparse.csc_matrix:\n    if not 0 < density < 1:\n        raise ValueError("density must be between zero and one")\n    rng = np.random.default_rng(seed)\n    if family == "lca-random":\n        off_diagonal = sparse.random(\n            n,\n            n,\n            density=density,\n            format="csc",\n            random_state=rng,\n            data_rvs=lambda size: rng.uniform(0.01, 0.04, size),\n        )\n    elif family == "io-block":\n        if blocks < 1 or blocks > n:\n            raise ValueError("blocks must be between 1 and n")\n        block_sizes = np.full(blocks, n // blocks, dtype=int)\n        block_sizes[: n % blocks] += 1\n        local_density = min(1.0, density * 0.8 * blocks)\n        local = sparse.block_diag(\n            [\n                sparse.random(\n                    size,\n                    size,\n                    density=local_density,\n                    format="csc",\n                    random_state=rng,\n                    data_rvs=lambda count: rng.uniform(0.001, 0.02, count),\n                )\n                for size in block_sizes\n            ],\n            format="csc",\n        )\n        cross = sparse.random(\n            n,\n            n,\n            density=density * 0.2,\n            format="csc",\n            random_state=rng,\n            data_rvs=lambda size: rng.uniform(0.0001, 0.005, size),\n        )\n        off_diagonal = local + cross\n    elif family == "banded":\n        half_bandwidth = max(1, min(n - 1, round(density * n / 2)))\n        return banded_matrix(n, half_bandwidth, seed, diagonal_span)\n    else:\n        raise ValueError(f"Unknown matrix family: {family}")\n    off_diagonal.setdiag(0.0)\n    off_diagonal.eliminate_zeros()\n\n    # Normalize columns so the increasing density does not make A singular.\n    column_sums = np.asarray(off_diagonal.sum(axis=0)).ravel()\n    scale = np.ones(n)\n    nonzero = column_sums > 0\n    scale[nonzero] = np.minimum(1.0, 0.4 / column_sums[nonzero])\n    off_diagonal = off_diagonal @ sparse.diags(scale, format="csc")\n    matrix = sparse.eye(n, format="csc") - off_diagonal\n    row_scales = 10 ** rng.uniform(-diagonal_span / 2, diagonal_span / 2, size=n)\n    matrix = sparse.diags(row_scales, format="csc") @ matrix\n    matrix.sum_duplicates()\n    matrix.sort_indices()\n    return matrix\n\n\ndef matrix_fingerprint(matrix: sparse.csc_matrix) -> str:\n    digest = hashlib.blake2b(digest_size=12)\n    for array in (matrix.indptr, matrix.indices, matrix.data):\n        digest.update(np.ascontiguousarray(array).view(np.uint8))\n    return digest.hexdigest()\n\n\ndef solve(\n    matrix: sparse.csc_matrix,\n    demand: np.ndarray,\n    solver: str,\n    rtol: float,\n    restart: int,\n    maxiter: int,\n    x0: np.ndarray | None = None,\n) -> tuple[np.ndarray, int | None, int]:\n    if solver == "numpy-dense":\n        return np.linalg.solve(matrix.toarray(), demand), None, 0\n    if solver == "superlu":\n        return spsolve(matrix, demand, use_umfpack=False), None, 0\n    if solver == "umfpack":\n        import scikits.umfpack  # noqa: F401\n\n        return spsolve(matrix, demand, use_umfpack=True), None, 0\n    if solver == "pardiso":\n        from pypardiso import spsolve as pardiso_spsolve\n\n        return pardiso_spsolve(matrix, demand), None, 0\n\n    residual_history: list[float] = []\n    preconditioner = None\n    if solver == "jacobi-gmres":\n        diagonal = matrix.diagonal()\n        if np.any(diagonal == 0):\n            raise ValueError("Jacobi requires a non-zero matrix diagonal")\n        inverse_diagonal = 1.0 / diagonal\n        preconditioner = LinearOperator(\n            matrix.shape,\n            matvec=lambda vector: inverse_diagonal * vector,\n            dtype=matrix.dtype,\n        )\n    elif solver != "gmres":\n        raise ValueError(f"Unknown solver: {solver}")\n\n    solution, info = gmres(\n        matrix,\n        demand,\n        x0=x0,\n        M=preconditioner,\n        rtol=rtol,\n        atol=0.0,\n        restart=restart,\n        maxiter=maxiter,\n        callback=residual_history.append,\n        callback_type="pr_norm",\n    )\n    return solution, int(info), len(residual_history)\n\n\ndef solve_many(\n    matrix: sparse.csc_matrix,\n    demands: list[np.ndarray],\n    solver: str,\n    rtol: float,\n    restart: int,\n    maxiter: int,\n) -> tuple[list[np.ndarray], int | None, list[int], float, float, float | None]:\n    """Solve multiple right-hand sides and separate setup from repeated solves."""\n    factorization_seconds = 0.0\n    fill_ratio = None\n    info: int | None = None\n    iterations: list[int] = []\n\n    if solver == "numpy-dense":\n        dense = matrix.toarray()\n        started = time.perf_counter()\n        solutions = [np.linalg.solve(dense, demand) for demand in demands]\n        return (\n            solutions,\n            None,\n            [0] * len(demands),\n            0.0,\n            time.perf_counter() - started,\n            None,\n        )\n\n    if solver == "superlu":\n        started = time.perf_counter()\n        factor = splu(matrix)\n        factorization_seconds = time.perf_counter() - started\n        fill_ratio = float((factor.L.nnz + factor.U.nnz) / matrix.nnz)\n        started = time.perf_counter()\n        solutions = [factor.solve(demand) for demand in demands]\n        rhs_seconds = time.perf_counter() - started\n        return (\n            solutions,\n            None,\n            [0] * len(demands),\n            factorization_seconds,\n            rhs_seconds,\n            fill_ratio,\n        )\n\n    if solver == "umfpack":\n        import scikits.umfpack as umfpack\n\n        context = umfpack.UmfpackContext()\n        started = time.perf_counter()\n        lower, upper, *_ = context.lu(matrix)\n        factorization_seconds = time.perf_counter() - started\n        fill_ratio = float((lower.nnz + upper.nnz) / matrix.nnz)\n        started = time.perf_counter()\n        solutions = [\n            context.solve(umfpack.UMFPACK_A, matrix, demand) for demand in demands\n        ]\n        rhs_seconds = time.perf_counter() - started\n        return (\n            solutions,\n            None,\n            [0] * len(demands),\n            factorization_seconds,\n            rhs_seconds,\n            fill_ratio,\n        )\n\n    if solver == "pardiso":\n        from pypardiso import spsolve as pardiso_spsolve\n\n        started = time.perf_counter()\n        solutions = [pardiso_spsolve(matrix, demand) for demand in demands]\n        rhs_seconds = time.perf_counter() - started\n        return solutions, None, [0] * len(demands), 0.0, rhs_seconds, None\n\n    solutions = []\n    started = time.perf_counter()\n    for demand in demands:\n        solution, current_info, current_iterations = solve(\n            matrix, demand, solver, rtol, restart, maxiter\n        )\n        solutions.append(solution)\n        iterations.append(current_iterations)\n        if current_info not in (None, 0):\n            info = current_info\n    return solutions, info or 0, iterations, 0.0, time.perf_counter() - started, None\n\n\ndef package_version(name: str) -> str | None:\n    try:\n        return importlib.metadata.version(name)\n    except importlib.metadata.PackageNotFoundError:\n        return None\n\n\ndef run(args: argparse.Namespace) -> dict[str, object]:\n    with MemoryMonitor() as generation_memory:\n        started = time.perf_counter()\n        if args.topology == "constant-degree":\n            matrix = constant_degree_matrix(\n                args.size, args.degree, args.seed, args.diagonal_span\n            )\n        elif args.topology == "banded":\n            matrix = banded_matrix(\n                args.size, args.degree, args.seed, args.diagonal_span\n            )\n        else:\n            matrix = fixed_density_matrix(\n                args.size,\n                args.density,\n                args.seed,\n                args.diagonal_span,\n                family=args.matrix_family,\n                blocks=args.blocks,\n            )\n        generation_seconds = time.perf_counter() - started\n\n    demands = []\n    for index in range(args.rhs_count):\n        demand = np.zeros(args.size)\n        demand[index % args.size] = 1.0\n        demands.append(demand)\n    matrix_bytes = matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes\n\n    with MemoryMonitor() as memory:\n        started = time.perf_counter()\n        (\n            solutions,\n            info,\n            iterations,\n            factorization_seconds,\n            rhs_solve_seconds,\n            fill_ratio,\n        ) = solve_many(\n            matrix,\n            demands,\n            args.solver,\n            args.rtol,\n            args.restart,\n            args.maxiter,\n        )\n        solve_seconds = time.perf_counter() - started\n\n    residuals = [\n        float(np.linalg.norm(matrix @ solution - demand) / np.linalg.norm(demand))\n        for solution, demand in zip(solutions, demands)\n    ]\n    relative_residual = max(residuals)\n    converged = info in (None, 0) and relative_residual <= max(args.rtol * 10, 1e-12)\n\n    return {\n        "kind": "synthetic",\n        "topology": args.topology,\n        "matrix_family": "banded" if args.topology == "banded" else args.matrix_family,\n        "blocks": args.blocks if args.matrix_family == "io-block" else None,\n        "solver": args.solver,\n        "size": args.size,\n        "shape": [args.size, args.size],\n        "degree": (\n            args.degree if args.topology in {"constant-degree", "banded"} else None\n        ),\n        "target_density": args.density if args.topology == "fixed-density" else None,\n        "diagonal_span_orders": args.diagonal_span,\n        "nnz": int(matrix.nnz),\n        "density": float(matrix.nnz / (args.size * args.size)),\n        "matrix_storage_bytes": int(matrix_bytes),\n        "matrix_fingerprint": matrix_fingerprint(matrix),\n        "seed": args.seed,\n        "rtol": args.rtol,\n        "restart": args.restart,\n        "maxiter": args.maxiter,\n        "generation_seconds": generation_seconds,\n        "generation_incremental_peak_rss_bytes": generation_memory.incremental_peak_bytes,\n        "solve_seconds": solve_seconds,\n        "factorization_seconds": factorization_seconds,\n        "rhs_solve_seconds": rhs_solve_seconds,\n        "rhs_count": args.rhs_count,\n        "seconds_per_rhs": rhs_solve_seconds / args.rhs_count,\n        "lu_fill_ratio": fill_ratio,\n        "baseline_rss_bytes": memory.baseline_bytes,\n        "peak_rss_bytes": memory.peak_bytes,\n        "incremental_peak_rss_bytes": memory.incremental_peak_bytes,\n        "iterations": max(iterations, default=0),\n        "iterations_per_rhs": iterations,\n        "solver_info": info,\n        "relative_residual": relative_residual,\n        "converged": converged,\n        "environment": {\n            "python": platform.python_version(),\n            "platform": platform.platform(),\n            "processor": platform.processor(),\n            "numpy": np.__version__,\n            "scipy": scipy.__version__,\n            "scikit_umfpack": package_version("scikit-umfpack"),\n            "physical_memory_bytes": psutil.virtual_memory().total,\n        },\n    }\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\n        "--solver",\n        choices=(\n            "numpy-dense",\n            "superlu",\n            "umfpack",\n            "pardiso",\n            "gmres",\n            "jacobi-gmres",\n        ),\n        required=True,\n    )\n    parser.add_argument(\n        "--topology",\n        choices=("constant-degree", "fixed-density", "banded"),\n        default="constant-degree",\n    )\n    parser.add_argument("--size", type=int, required=True)\n    parser.add_argument("--degree", type=int, default=8)\n    parser.add_argument("--density", type=float, default=0.001)\n    parser.add_argument(\n        "--matrix-family",\n        choices=("lca-random", "io-block", "banded"),\n        default="lca-random",\n    )\n    parser.add_argument("--blocks", type=int, default=8)\n    parser.add_argument("--rhs-count", type=int, default=1)\n    parser.add_argument("--diagonal-span", type=float, default=4.0)\n    parser.add_argument("--seed", type=int, default=2026)\n    parser.add_argument("--rtol", type=float, default=1e-4)\n    parser.add_argument("--restart", type=int, default=50)\n    parser.add_argument("--maxiter", type=int, default=300)\n    args = parser.parse_args()\n    print(json.dumps(run(args), sort_keys=True))\n\n\nif __name__ == "__main__":\n    main()\n')
_embedded_suite.write_text('"""Orchestrate isolated synthetic solver benchmarks with optional time guards."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\n\ndef run_worker(\n    python: str,\n    worker: Path,\n    solver: str,\n    size: int,\n    topology: str,\n    degree: int,\n    density: float,\n    diagonal_span: float,\n    rtol: float,\n    matrix_family: str,\n    blocks: int,\n    rhs_count: int,\n    timeout_seconds: float | None,\n) -> dict[str, Any]:\n    command = [\n        python,\n        str(worker),\n        "--solver",\n        solver,\n        "--size",\n        str(size),\n        "--topology",\n        topology,\n        "--degree",\n        str(degree),\n        "--density",\n        str(density),\n        "--diagonal-span",\n        str(diagonal_span),\n        "--rtol",\n        str(rtol),\n        "--matrix-family",\n        matrix_family,\n        "--blocks",\n        str(blocks),\n        "--rhs-count",\n        str(rhs_count),\n    ]\n    started = time.perf_counter()\n    try:\n        completed = subprocess.run(\n            command,\n            check=True,\n            capture_output=True,\n            text=True,\n            timeout=timeout_seconds,\n        )\n    except subprocess.TimeoutExpired:\n        return {\n            "kind": "synthetic",\n            "solver": solver,\n            "size": size,\n            "topology": topology,\n            "degree": degree if topology == "constant-degree" else None,\n            "target_density": density if topology == "fixed-density" else None,\n            "diagonal_span_orders": diagonal_span,\n            "rtol": rtol,\n            "matrix_family": matrix_family,\n            "rhs_count": rhs_count,\n            "timed_out": True,\n            "status": "TIMEOUT",\n            "worker_wall_seconds": time.perf_counter() - started,\n        }\n    except subprocess.CalledProcessError as error:\n        return {\n            "kind": "synthetic",\n            "solver": solver,\n            "size": size,\n            "topology": topology,\n            "degree": degree if topology == "constant-degree" else None,\n            "target_density": density if topology == "fixed-density" else None,\n            "diagonal_span_orders": diagonal_span,\n            "rtol": rtol,\n            "matrix_family": matrix_family,\n            "rhs_count": rhs_count,\n            "timed_out": False,\n            "status": "FAILED",\n            "worker_wall_seconds": time.perf_counter() - started,\n            "error": error.stderr[-2000:],\n        }\n\n    result = json.loads(completed.stdout.strip().splitlines()[-1])\n    result["timed_out"] = False\n    result["status"] = "COMPLETED"\n    result["worker_wall_seconds"] = time.perf_counter() - started\n    return result\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--python", default=sys.executable)\n    parser.add_argument(\n        "--worker", type=Path, default=Path("benchmark_synthetic.py")\n    )\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--sizes", type=int, nargs="+", default=[500, 1000, 2500, 5000])\n    parser.add_argument(\n        "--solvers",\n        nargs="+",\n        default=[\n            "numpy-dense",\n            "superlu",\n            "umfpack",\n            "pardiso",\n            "gmres",\n            "jacobi-gmres",\n        ],\n    )\n    parser.add_argument(\n        "--topology",\n        choices=("constant-degree", "fixed-density", "banded"),\n        default="constant-degree",\n    )\n    parser.add_argument("--degree", type=int, default=8)\n    parser.add_argument("--density", type=float, default=0.001)\n    parser.add_argument("--degrees", type=int, nargs="+")\n    parser.add_argument("--densities", type=float, nargs="+")\n    parser.add_argument(\n        "--matrix-family",\n        choices=("lca-random", "io-block", "banded"),\n        default="lca-random",\n    )\n    parser.add_argument("--blocks", type=int, default=8)\n    parser.add_argument("--rhs-count", type=int, default=1)\n    parser.add_argument("--rhs-counts", type=int, nargs="+")\n    parser.add_argument("--diagonal-span", type=float, default=4.0)\n    parser.add_argument("--rtol", type=float, default=1e-4)\n    parser.add_argument("--dense-max", type=int, default=2500)\n    parser.add_argument("--run-timeout", type=float)\n    parser.add_argument("--total-budget", type=float)\n    parser.add_argument(\n        "--max-estimated-construction-mib",\n        type=float,\n        help="Skip a matrix before construction when estimated working memory exceeds this cap",\n    )\n    parser.add_argument(\n        "--construction-memory-multiplier",\n        type=float,\n        default=3.0,\n        help="Multiplier applied to estimated CSC storage for construction intermediates",\n    )\n    args = parser.parse_args()\n\n    results: list[dict[str, Any]] = []\n    suite_started = time.perf_counter()\n    stop = False\n    degrees = args.degrees or [args.degree]\n    densities = args.densities or [args.density]\n    rhs_counts = args.rhs_counts or [args.rhs_count]\n    cases = [\n        (size, degree, density, rhs_count)\n        for size in args.sizes\n        for degree in (degrees if args.topology == "constant-degree" else [args.degree])\n        for density in (\n            densities if args.topology == "fixed-density" else [args.density]\n        )\n        for rhs_count in rhs_counts\n    ]\n    for size, degree, density, rhs_count in cases:\n        estimated_nnz = (\n            size * (degree + 1)\n            if args.topology == "constant-degree"\n            else (\n                size * (2 * degree + 1)\n                if args.topology == "banded"\n                else int(size * size * density) + size\n            )\n        )\n        estimated_storage_bytes = estimated_nnz * 12 + (size + 1) * 4\n        estimated_construction_mib = (\n            estimated_storage_bytes * args.construction_memory_multiplier / 2**20\n        )\n        guarded = (\n            args.max_estimated_construction_mib is not None\n            and estimated_construction_mib > args.max_estimated_construction_mib\n        )\n        for solver in args.solvers:\n            if solver == "numpy-dense" and size > args.dense_max:\n                continue\n            if guarded:\n                result = {\n                    "kind": "synthetic",\n                    "solver": solver,\n                    "size": size,\n                    "topology": args.topology,\n                    "degree": (\n                        degree\n                        if args.topology in {"constant-degree", "banded"}\n                        else None\n                    ),\n                    "target_density": (\n                        density if args.topology == "fixed-density" else None\n                    ),\n                    "matrix_family": (\n                        "banded" if args.topology == "banded" else args.matrix_family\n                    ),\n                    "rhs_count": rhs_count,\n                    "estimated_nnz": estimated_nnz,\n                    "estimated_construction_mib": estimated_construction_mib,\n                    "status": "SKIPPED",\n                    "skip_reason": "MEMORY GUARD",\n                    "timed_out": False,\n                }\n                results.append(result)\n                print(json.dumps(result), flush=True)\n                continue\n            if (\n                args.total_budget is not None\n                and time.perf_counter() - suite_started >= args.total_budget\n            ):\n                stop = True\n                break\n            result = run_worker(\n                args.python,\n                args.worker,\n                solver,\n                size,\n                args.topology,\n                degree,\n                density,\n                args.diagonal_span,\n                args.rtol,\n                args.matrix_family,\n                args.blocks,\n                rhs_count,\n                args.run_timeout,\n            )\n            result["estimated_nnz"] = estimated_nnz\n            result["estimated_construction_mib"] = estimated_construction_mib\n            results.append(result)\n            print(\n                json.dumps(\n                    {\n                        key: result.get(key)\n                        for key in (\n                            "solver",\n                            "size",\n                            "degree",\n                            "target_density",\n                            "matrix_family",\n                            "rhs_count",\n                            "solve_seconds",\n                            "factorization_seconds",\n                            "rhs_solve_seconds",\n                            "lu_fill_ratio",\n                            "incremental_peak_rss_bytes",\n                            "iterations",\n                            "relative_residual",\n                            "timed_out",\n                            "status",\n                            "error",\n                        )\n                    }\n                ),\n                flush=True,\n            )\n        if stop:\n            break\n\n    payload = {\n        "suite_wall_seconds": time.perf_counter() - suite_started,\n        "stopped_by_total_budget": stop,\n        "run_timeout_seconds": args.run_timeout,\n        "total_budget_seconds": args.total_budget,\n        "max_estimated_construction_mib": args.max_estimated_construction_mib,\n        "construction_memory_multiplier": args.construction_memory_multiplier,\n        "results": results,\n    }\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    args.output.write_text(json.dumps(payload, indent=2) + "\\n", encoding="utf-8")\n\n\nif __name__ == "__main__":\n    main()\n')


# Replace scipy.sparse.random with a direct CSC builder for large fixed-density cases.
# It allocates the final CSC arrays once and uses bounded temporary chunks.
_DIRECT_CSC_FIXED_DENSITY_SOURCE = r'''
_legacy_fixed_density_matrix = fixed_density_matrix

def fixed_density_matrix(
    n: int,
    density: float,
    seed: int,
    diagonal_span: float,
    family: str = "lca-random",
    blocks: int = 8,
) -> sparse.csc_matrix:
    if family != "lca-random":
        return _legacy_fixed_density_matrix(
            n, density, seed, diagonal_span, family=family, blocks=blocks
        )
    if not 0 < density < 1:
        raise ValueError("density must be between zero and one")

    rng = np.random.default_rng(seed)
    entries_per_column = max(1, min(n - 1, round(density * (n - 1))))
    stride = entries_per_column + 1  # off-diagonal entries plus the diagonal
    total_entries = n * stride
    index_dtype = np.int32 if total_entries <= np.iinfo(np.int32).max else np.int64
    indices = np.empty(total_entries, dtype=index_dtype)
    data = np.empty(total_entries, dtype=np.float64)
    indptr = np.arange(0, total_entries + 1, stride, dtype=index_dtype)
    row_scales = 10 ** rng.uniform(-diagonal_span / 2, diagonal_span / 2, size=n)

    # Limit temporary row/data arrays to roughly two million entries each.
    columns_per_chunk = max(1, min(n, 2_000_000 // entries_per_column))
    for start in range(0, n, columns_per_chunk):
        stop = min(n, start + columns_per_chunk)
        columns = np.arange(start, stop, dtype=np.int64)
        offsets = rng.choice(n - 1, size=entries_per_column, replace=False) + 1
        rows = (columns[:, None] + offsets[None, :]) % n
        coefficients = rng.uniform(0.01, 0.04, size=rows.shape)
        coefficients *= np.minimum(1.0, 0.4 / coefficients.sum(axis=1))[:, None]

        target = slice(start * stride, stop * stride)
        target_indices = indices[target].reshape(stop - start, stride)
        target_data = data[target].reshape(stop - start, stride)
        target_indices[:, :-1] = rows
        target_data[:, :-1] = -coefficients * row_scales[rows]
        target_indices[:, -1] = columns
        target_data[:, -1] = row_scales[columns]

    matrix = sparse.csc_matrix(
        (data, indices, indptr), shape=(n, n), copy=False
    )
    matrix.sort_indices()
    return matrix
'''
_worker_source = _embedded_worker.read_text()
_worker_source = _worker_source.replace(
    "\ndef matrix_fingerprint(matrix: sparse.csc_matrix) -> str:",
    _DIRECT_CSC_FIXED_DENSITY_SOURCE
    + "\ndef matrix_fingerprint(matrix: sparse.csc_matrix) -> str:",
    1,
)
_embedded_worker.write_text(_worker_source)


def run_benchmark(
    name,
    *,
    sizes,
    solvers,
    topology="constant-degree",
    degree=8,
    degrees=None,
    densities=None,
    rhs_counts=None,
    matrix_family="lca-random",
    blocks=8,
    rtol=1e-4,
    worker_timeout=None,
    total_budget=None,
    memory_guard_mib=None,
    construction_memory_multiplier=None,
    suite_timeout=None,
):
    # Run an isolated benchmark grid and return its records.
    output = _embedded_dir / f"{name}.json"
    command = [
        sys.executable,
        str(_embedded_suite),
        "--python", sys.executable,
        "--worker", str(_embedded_worker),
        "--output", str(output),
        "--sizes", *map(str, sizes),
        "--solvers", *solvers,
        "--topology", topology,
        "--degree", str(degree),
        "--matrix-family", matrix_family,
        "--blocks", str(blocks),
        "--rtol", str(rtol),
    ]
    optional_lists = {
        "--degrees": degrees,
        "--densities": densities,
        "--rhs-counts": rhs_counts,
    }
    for flag, values in optional_lists.items():
        if values:
            command.extend([flag, *map(str, values)])
    optional_values = {
        "--run-timeout": worker_timeout,
        "--total-budget": total_budget,
        "--max-estimated-construction-mib": memory_guard_mib,
        "--construction-memory-multiplier": construction_memory_multiplier,
    }
    for flag, value in optional_values.items():
        if value is not None:
            command.extend([flag, str(value)])
    subprocess.run(command, check=True, capture_output=True, text=True, timeout=suite_timeout)
    return json.loads(output.read_text())


In [ ]:
from __future__ import annotations

import importlib.metadata
import sys
from time import perf_counter

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
import psutil
import scipy
from IPython.display import Markdown, display

SEED = 2026
RTOL = 1e-4
available_mib = psutil.virtual_memory().available / 2**20
memory_guard_mib = int(min(8_192, max(512, available_mib * 0.25)))
try:
    import pypardiso
    PARDISO_AVAILABLE = True
except ImportError:
    PARDISO_AVAILABLE = False
DIRECT_SOLVERS = ["umfpack"] + (["pardiso"] if PARDISO_AVAILABLE else [])
colors = {
    "numpy-dense": "#7F8C8D", "superlu": "#C44E52", "umfpack": "#DD8452",
    "pardiso": "#937860", "gmres": "#4C72B0", "jacobi-gmres": "#12A594",
    "jacobi-gmres-no-guess": "#8172B2",
}
solver_names = {
    "numpy-dense": "NumPy dense",
    "superlu": "SuperLU",
    "umfpack": "UMFPACK",
    "pardiso": "Pardiso",
    "gmres": "GMRES",
    "jacobi-gmres": "Jacobi + GMRES",
    "jacobi-gmres-no-guess": "Jacobi + GMRES (no warm start)",
}

def observed_log_ratio_scale(values):
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    limit = max(float(np.max(np.abs(finite))), 0.25) if finite.size else 1.0
    ticks = np.linspace(-limit, limit, 5)
    labels = [
        f"{10 ** tick:,.0f}×" if 10 ** tick >= 1 else f"{10 ** tick:.2g}×"
        for tick in ticks
    ]
    return limit, ticks, labels

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "figure.dpi": 120,
})


{
    "benchmark mode": "live, self-contained notebook",
    "worker Python": sys.executable,
    "NumPy": np.__version__,
    "SciPy": scipy.__version__,
    "scikit-umfpack": importlib.metadata.version("scikit-umfpack"),
}


## Solver toolbox

- **NumPy dense** stores and solves the complete matrix, including all its zeros.
- **SuperLU** is SciPy's general sparse direct solver.
- **UMFPACK** is a sparse direct solver designed to limit unnecessary work and memory.
- **GMRES** approaches the answer iteratively, stopping when the requested tolerance is reached.
- **Jacobi + GMRES** rescales the equations using the diagonal before iterating.
- **Pardiso**, when installed, appears as another high-performance sparse direct solver.

Every comparison also checks convergence and the relative residual $||Ax-b||/||b||$.


# 1 · Scale the matrix, isolate every solver

Each solver runs in a fresh subprocess. This prevents one factorisation from contaminating the next solver's peak memory. A 2 ms RSS sampler captures allocations made by NumPy, SciPy, UMFPACK, and BLAS—not only Python objects. Incremental RSS is measured above the post-matrix-construction baseline.

Main series: approximately eight inputs per activity, from 500 to 50,000 activities. Dense
solving stops at 2,500; every other worker has a visible 12-second timeout.


In [ ]:
synthetic_payload = run_benchmark(
    "scaling",
    sizes=[500, 1_000, 2_500, 5_000, 10_000, 20_000, 50_000],
    solvers=["numpy-dense", "jacobi-gmres", *DIRECT_SOLVERS, "gmres", "superlu"],
    rtol=RTOL,
    worker_timeout=12,
    total_budget=65,
    suite_timeout=82,
)
synthetic = pd.json_normalize(synthetic_payload["results"])
synthetic["memory_mib"] = synthetic["incremental_peak_rss_bytes"] / 2**20


In [ ]:
solver_order = ["numpy-dense", "superlu", *DIRECT_SOLVERS, "gmres", "jacobi-gmres"]
colors = {
    "numpy-dense": "#7F8C8D",
    "superlu": "#C44E52",
    "umfpack": "#DD8452",
    "pardiso": "#937860",
    "gmres": "#4C72B0",
    "jacobi-gmres": "#12A594",
    "jacobi-gmres-no-guess": "#8172B2",
}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
valid = synthetic[synthetic["solve_seconds"].notna()]
for solver in solver_order:
    subset = valid[valid.solver == solver].sort_values("size")
    if subset.empty:
        continue
    label = solver_names[solver]
    axes[0].plot(subset["size"], subset["solve_seconds"], "o-", label=label, color=colors[solver])
    axes[1].plot(subset["size"], subset["memory_mib"], "o-", label=label, color=colors[solver])

#axes[0].set_yscale("log")
axes[0].set_ylabel("solve time [s]")
#axes[1].set_yscale("log")
axes[1].set_ylabel("additional peak memory [MiB]")
for axis in axes:
    axis.set_xlabel("matrix rows / columns")
    axis.grid(alpha=0.25, which="both")
axes[0].set_title("Runtime separates as the system grows")
axes[1].set_title("Direct factorization can increase memory")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False, fontsize=8)
fig.tight_layout(rect=[0, 0.12, 1, 1])
plt.show()

not_completed_scaling = synthetic[synthetic.status != "COMPLETED"].copy()
if not not_completed_scaling.empty:
    error_text = (
        not_completed_scaling["error"].fillna("")
        if "error" in not_completed_scaling.columns
        else pd.Series("", index=not_completed_scaling.index)
    )
    oom = error_text.str.contains("out_of_memory", case=False)
    not_completed_scaling.loc[oom, "status"] = "OUT OF MEMORY"
    display(not_completed_scaling[["solver", "size", "status"]].style.hide(axis="index"))


### Density versus size

**Matrix density** is the share of matrix entries that are nonzero. This sweep varies both system
size and density from 0.1% to 5%. The later stress test covers 3% to 15% and
shows which combinations exceed the memory guard.

Each heatmap cell answers one question:
**how many times faster is one solver than the other?**


In [ ]:
density_size_payload = run_benchmark(
    "density versus size",
    sizes=[1_000, 2_500, 5_000, 7_500],
    solvers=["jacobi-gmres", *DIRECT_SOLVERS],
    topology="fixed-density",
    densities=[0.001, 0.005, 0.01, 0.03, 0.05],
    rtol=RTOL,
    worker_timeout=12,
    total_budget=45,
    memory_guard_mib=memory_guard_mib,
    suite_timeout=62,
)
density_size = pd.json_normalize(density_size_payload["results"])
density_size["speedup"] = np.nan
for (size, density), subset in density_size.groupby(["size", "target_density"]):
    direct = subset.loc[subset.solver == "umfpack", "solve_seconds"]
    iterative = subset.loc[subset.solver == "jacobi-gmres", "solve_seconds"]
    if len(direct) == len(iterative) == 1:
        density_size.loc[subset.index, "speedup"] = direct.iloc[0] / iterative.iloc[0]

heat = density_size[density_size.solver == "jacobi-gmres"].pivot(
    index="target_density", columns="size", values="speedup"
)
log_heat = np.log10(heat)
heat_limit, heat_ticks, heat_tick_labels = observed_log_ratio_scale(log_heat)
fig, axis = plt.subplots(figsize=(8.5, 4.5))
image = axis.imshow(
    log_heat, aspect="auto", origin="lower",
    cmap="PiYG", vmin=-heat_limit, vmax=heat_limit,
)
axis.set_xticks(range(len(heat.columns)), [f"{n:,}" for n in heat.columns])
axis.set_yticks(range(len(heat.index)), [f"{density:.1%}" for density in heat.index])
axis.set_xlabel("matrix size"); axis.set_ylabel("matrix density")
axis.set_title("UMFPACK time ÷ Jacobi + GMRES time (>1× favors Jacobi)")
for row, density in enumerate(heat.index):
    for column, size in enumerate(heat.columns):
        value = heat.loc[density, size]
        if np.isfinite(value):
            label = f"{value:.0f}×" if value >= 1 else "<1×"
            text_color = "white" if abs(np.log10(value)) > 0.45 * heat_limit else "black"
            axis.text(
                column, row, label,
                ha="center", va="center", fontsize=8,
                color=text_color, fontweight="bold",
            )
bar = fig.colorbar(image, ax=axis, label="time ratio (scaled to observed maximum)")
bar.set_ticks(heat_ticks); bar.set_ticklabels(heat_tick_labels)
fig.tight_layout(); plt.show()


## Guarded size–density stress test

With constant density, nonzeros grow with $n^2$, not $n$. Curves compare 2,000, 6,000, and
15,000 rows from 0.1% to 15% density. A memory guard marks unsafe combinations **SKIPPED**
before allocation; solver-level failures remain labelled in the heatmap.


In [ ]:
density_payload = run_benchmark(
    "density grid",
    sizes=[2_000, 6_000, 15_000],
    solvers=["jacobi-gmres", *DIRECT_SOLVERS],
    topology="fixed-density",
    densities=[0.001, 0.03, 0.05, 0.075, 0.1, 0.15],
    rtol=RTOL,
    worker_timeout=60,
    total_budget=120,
    memory_guard_mib=memory_guard_mib,
    suite_timeout=185,
)
density_results = pd.json_normalize(density_payload["results"])
density_results["incremental_peak_MiB"] = density_results["incremental_peak_rss_bytes"] / 2**20
density_results["runtime_status"] = density_results.get("status", "COMPLETED")

completed = density_results[density_results.runtime_status == "COMPLETED"].copy()
completed["log10_speedup_umfpack_over_jacobi"] = np.nan
for (size, density), subset in completed.groupby(["size", "target_density"]):
    direct = subset.loc[subset.solver == "umfpack", "solve_seconds"]
    iterative = subset.loc[subset.solver == "jacobi-gmres", "solve_seconds"]
    if len(direct) == 1 and len(iterative) == 1:
        value = np.log10(direct.iloc[0] / iterative.iloc[0])
        completed.loc[subset.index, "log10_speedup_umfpack_over_jacobi"] = value

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
size_styles = {2_000: ("-", "o"), 6_000: ("--", "s"), 15_000: (":", "^")}
for (solver, size), subset in completed.groupby(["solver", "size"]):
    subset = subset.sort_values("density")
    line_style, marker = size_styles[size]
    axes[0].plot(
        subset["density"], subset["solve_seconds"],
        linestyle=line_style, marker=marker,
        label=f"{solver_names[solver]}, n={size:,}", color=colors[solver],
    )
axes[0].set_xlabel("realized matrix density")
axes[0].set_ylabel("solve time [s]")
#axes[0].set_yscale("log")
axes[0].xaxis.set_major_formatter(PercentFormatter(1.0))
axes[0].set_title("Density effect at n = 2k, 6k, and 15k")
axes[0].grid(alpha=0.25, which="both")
axes[0].legend(frameon=False, fontsize=7, ncol=2)

size_order = sorted(density_results["size"].unique())
density_order = sorted(density_results["target_density"].unique())
pivot = completed[completed.solver == "jacobi-gmres"].pivot_table(
    index="size", columns="target_density", values="log10_speedup_umfpack_over_jacobi"
).reindex(index=size_order, columns=density_order)
stress_limit, stress_ticks, stress_tick_labels = observed_log_ratio_scale(pivot)
image = axes[1].imshow(
    pivot, aspect="auto", origin="lower",
    cmap="PiYG", vmin=-stress_limit, vmax=stress_limit,
)
axes[1].set_xticks(range(len(pivot.columns)), [f"{value:.1%}" for value in pivot.columns], rotation=45, ha="right")
axes[1].set_yticks(range(len(pivot.index)), pivot.index)
axes[1].set_xlabel("target density")
axes[1].set_ylabel("matrix size")
axes[1].set_title("Where does Jacobi + GMRES become faster? (>1×)")
for row, size in enumerate(pivot.index):
    for column, density in enumerate(pivot.columns):
        value = pivot.loc[size, density]
        if np.isfinite(value):
            ratio = 10 ** value
            label = f"{ratio:.0f}×" if ratio >= 1 else "<1×"
            color = "white" if abs(value) > 0.45 * stress_limit else "black"
        else:
            direct_case = density_results[
                (density_results["size"] == size)
                & (density_results["target_density"] == density)
                & (density_results["solver"] == "umfpack")
            ]
            status = direct_case["runtime_status"].iloc[0] if len(direct_case) else ""
            error = str(direct_case["error"].iloc[0]) if len(direct_case) and "error" in direct_case else ""
            label = "OOM" if "out_of_memory" in error else status
            color = "black"
        if label:
            axes[1].text(
                column, row, label,
                ha="center", va="center", fontsize=7,
                color=color, fontweight="bold",
            )
bar = fig.colorbar(image, ax=axes[1], label="UMFPACK time ÷ Jacobi time (observed scale)")
bar.set_ticks(stress_ticks); bar.set_ticklabels(stress_tick_labels)
fig.tight_layout()
plt.show()


### Large synthetic systems

This sweep tests 10,000, 30,000, 60,000, 100,000, and 300,000 rows. The solver comparison uses
**1% target density**; a separate Jacobi sweep measures worker peak RSS at **0.5%, 1%, 5%, and
10%**. A direct CSC constructor reduces temporary copies; unsafe cases remain **SKIPPED**.


In [ ]:
large_sizes = [10_000, 30_000, 60_000, 100_000, 300_000]
large_jacobi_scale_payload = run_benchmark(
    "large Jacobi systems at 1 percent",
    sizes=[10_000, 30_000, 60_000, 100_000, 300_000],
    solvers=["jacobi-gmres"],
    topology="fixed-density",
    densities=[0.01],
    rtol=RTOL,
    worker_timeout=120,
    total_budget=150,
    memory_guard_mib=4_096,
    construction_memory_multiplier=3.0,
    suite_timeout=275,
)
large_direct_scale_payload = run_benchmark(
    "large direct systems at 1 percent",
    sizes=large_sizes,
    solvers=DIRECT_SOLVERS,
    topology="fixed-density",
    densities=[0.01],
    rtol=RTOL,
    worker_timeout=60,
    total_budget=30,
    memory_guard_mib=1_200,
    construction_memory_multiplier=3.0,
    suite_timeout=92,
)
large_memory_guard_mib = 2_000
large_memory_payload = run_benchmark(
    "large systems memory by density",
    sizes=large_sizes,
    solvers=["jacobi-gmres"],
    topology="fixed-density",
    densities=[0.005, 0.05, 0.1],
    rtol=RTOL,
    worker_timeout=120,
    total_budget=150,
    memory_guard_mib=large_memory_guard_mib,
    construction_memory_multiplier=3.0,
    suite_timeout=275,
)
large_jacobi_scale = pd.json_normalize(large_jacobi_scale_payload["results"])
large_direct_scale = pd.json_normalize(large_direct_scale_payload["results"])
large_scale = pd.concat([large_jacobi_scale, large_direct_scale], ignore_index=True)
large_memory = pd.json_normalize(large_memory_payload["results"])
completed_scale = large_scale[large_scale.status == "COMPLETED"].copy()
completed_memory = large_memory[large_memory.status == "COMPLETED"].copy()
memory_plot = pd.concat([
    completed_memory,
    completed_scale[
        (completed_scale["solver"] == "jacobi-gmres")
        & np.isclose(completed_scale["target_density"], 0.01)
    ],
], ignore_index=True).drop_duplicates(["size", "target_density"], keep="last")
memory_plot["peak_rss_mib"] = memory_plot["peak_rss_bytes"] / 2**20
assert completed_scale["solve_seconds"].dropna().le(60).all()
assert completed_memory["solve_seconds"].dropna().le(60).all()
size_position = {size: index for index, size in enumerate(large_sizes)}
density_colors = {0.005: "#4C72B0", 0.01: "#55A868", 0.05: "#C44E52", 0.1: "#8172B2"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for solver, subset in completed_scale.groupby("solver"):
    subset = subset.sort_values("size")
    label = solver_names[solver]
    axes[0].plot(
        [size_position[size] for size in subset["size"]], subset["solve_seconds"],
        "o-", label=label, color=colors[solver],
    )
for density, subset in memory_plot.groupby("target_density"):
    subset = subset.sort_values("size")
    axes[1].plot(
        [size_position[size] for size in subset["size"]], subset["peak_rss_mib"],
        "o-", label=f"{density:.1%}", color=density_colors[density],
    )
for axis in axes:
    axis.set_xlabel("matrix size"); axis.grid(alpha=0.25)
    axis.set_xticks(range(len(large_sizes)), [f"{size:,}" for size in large_sizes], rotation=25)
axes[0].set_ylabel("solve time [s]"); axes[1].set_ylabel("worker peak RSS [MiB]")
axes[0].set_title("Solve time at 1% target density"); axes[1].set_title("Jacobi peak memory across densities")
axes[0].legend(frameon=False, fontsize=8)
axes[1].legend(frameon=False, fontsize=8, ncol=2, title="target density")
fig.tight_layout(); plt.show()

large_status = pd.concat([
    large_jacobi_scale.assign(suite="1% Jacobi comparison"),
    large_direct_scale.assign(suite="1% direct comparison"),
    large_memory.assign(suite="Jacobi density sweep"),
], ignore_index=True)
not_completed = large_status[large_status.status != "COMPLETED"].copy()
if not not_completed.empty:
    error_text = not_completed["error"].fillna("") if "error" in not_completed else pd.Series("", index=not_completed.index)
    oom = error_text.str.contains("out_of_memory", case=False)
    not_completed.loc[oom, "status"] = "OUT OF MEMORY"
    failed = not_completed[not_completed["status"] != "SKIPPED"]
    if not failed.empty:
        display(failed[["suite", "solver", "size", "target_density", "status"]].style.hide(axis="index"))
    skipped = not_completed[not_completed["status"] == "SKIPPED"]
    if not skipped.empty:
        skipped_sizes = ", ".join(f"{size:,}" for size in sorted(skipped["size"].unique()))
        display(Markdown(f"**Memory guard:** {len(skipped)} solver cases skipped before allocation at sizes {skipped_sizes}."))


# 2 · Fixed matrix, many right-hand sides

One fixed 40,000 × 40,000 matrix can serve many demands. The left plot separates the one-time
factorization from the first solve. The right plot then shows total time as more demands reuse that
same matrix.


In [ ]:
rhs_payload = run_benchmark(
    "fixed matrix demands",
    sizes=[40_000],
    solvers=[*DIRECT_SOLVERS, "jacobi-gmres"],
    topology="banded",
    degree=30,
    rhs_counts=[1, 3, 10, 25, 50, 100, 250],
    rtol=RTOL,
    worker_timeout=15,
    total_budget=50,
    suite_timeout=70,
)
rhs = pd.json_normalize(rhs_payload["results"])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

first_demand = rhs[
    (rhs.rhs_count == 1) & rhs.solver.isin(["umfpack", "jacobi-gmres"])
].set_index("solver")
bar_solvers = [solver for solver in ["umfpack", "jacobi-gmres"] if solver in first_demand.index]
x = np.arange(len(bar_solvers))
setup_time = [first_demand.loc[solver, "factorization_seconds"] for solver in bar_solvers]
solve_time = [first_demand.loc[solver, "rhs_solve_seconds"] for solver in bar_solvers]
axes[0].bar(x, setup_time, label="one-time factorization", color="#B0B0B0")
axes[0].bar(x, solve_time, bottom=setup_time, label="solve first demand", color=[colors[solver] for solver in bar_solvers])
axes[0].set_xticks(x, [solver_names[solver] for solver in bar_solvers])
axes[0].set_ylabel("time [s]")
axes[0].set_title("Cost of the first demand")
axes[0].legend(frameon=False, fontsize=8)

for solver, subset in rhs.groupby("solver"):
    axes[1].plot(
        subset["rhs_count"], subset["solve_seconds"], "o-",
        label=solver_names[solver], color=colors[solver],
    )

comparison = rhs[rhs.solver.isin(["umfpack", "jacobi-gmres"])].pivot(
    index="rhs_count", columns="solver", values="solve_seconds"
)
if {"umfpack", "jacobi-gmres"}.issubset(comparison.columns):
    direct_wins = comparison.index[comparison["umfpack"] < comparison["jacobi-gmres"]]
    if len(direct_wins):
        crossover = direct_wins.min()
        axes[1].axvline(crossover, color="black", linestyle="--", linewidth=1)
        axes[1].text(crossover, axes[1].get_ylim()[1], f"  direct faster from ~{crossover:,}", va="top", fontsize=8)

axes[1].set_xlabel("demands solved with one fixed matrix")
axes[1].set_ylabel("total solve time [s]")
axes[1].set_title("Factorization reuse changes the winner")
axes[1].legend(frameon=False, fontsize=8)
for axis in axes:
    axis.grid(alpha=0.25)
fig.tight_layout(); plt.show()


# 3 · Changing matrix, repeated solves

This is a 500-iteration synthetic **Monte Carlo** analogue: the technosphere matrix changes slightly
at every sample. UMFPACK must refactorize each matrix. Jacobi + GMRES is shown both with a warm start
from the previous solution and without one, matching the role of `use_guess` in `bw2calc`.


In [ ]:
demand = np.zeros(10000)
demand[0] = 1.0
records = []
coupling_rng = np.random.default_rng(SEED)
sample_couplings = np.clip(0.8 + coupling_rng.normal(0, 0.015, 500), 0.7, 0.9)
solver_cases = [
    *((solver, solver, False) for solver in DIRECT_SOLVERS),
    ("jacobi-gmres", "jacobi-gmres", True),
    ("jacobi-gmres-no-guess", "jacobi-gmres", False),
]

for label, solver, reuse_guess in solver_cases:
    started = perf_counter()
    residuals = []
    iterations = []
    previous_solution = None
    for sample in range(500):
        matrix = banded_matrix(
            10_000, 4, SEED, 4.0,
            coupling=float(sample_couplings[sample]),
        )
        solve_started = perf_counter()
        solution, info, count = solve(
            matrix, demand, solver, RTOL, 50, 300,
            x0=previous_solution if reuse_guess else None,
        )
        elapsed = perf_counter() - solve_started
        residual = float(np.linalg.norm(matrix @ solution - demand) / np.linalg.norm(demand))
        if reuse_guess:
            previous_solution = solution
        residuals.append(residual)
        iterations.append(count)
        records.append({"solver": label, "sample": sample, "solve_seconds": elapsed, "iterations": count, "relative_residual": residual, "info": info})
    records.append({"solver": label, "sample": "TOTAL", "solve_seconds": perf_counter() - started, "iterations": int(np.median(iterations)), "relative_residual": max(residuals), "info": None})
changing = pd.DataFrame(records)
per_sample = changing[changing["sample"] != "TOTAL"].copy()
per_sample["cumulative_seconds"] = per_sample.groupby("solver")["solve_seconds"].cumsum()
per_sample["monte_carlo_iteration"] = per_sample["sample"].astype(int) + 1
mc_summary = (
    per_sample.groupby("solver")
    .agg(
        total_seconds=("solve_seconds", "sum"),
        median_iterations=("iterations", "median"),
        max_relative_residual=("relative_residual", "max"),
        failed_solves=("info", lambda values: int(values.dropna().ne(0).sum())),
    )
    .reset_index()
)
assert mc_summary["failed_solves"].eq(0).all()
assert mc_summary["max_relative_residual"].le(RTOL * 1.01).all()
display(
    mc_summary.style.format(
        {"total_seconds": "{:.2f}", "median_iterations": "{:.0f}", "max_relative_residual": "{:.2e}"}
    ).hide(axis="index")
)

fig, axis = plt.subplots(figsize=(8.5, 4.5))
for solver, subset in per_sample.groupby("solver"):
    label = "Jacobi + GMRES (warm start)" if solver == "jacobi-gmres" else solver_names[solver]
    axis.plot(
        subset["monte_carlo_iteration"],
        subset["cumulative_seconds"],
        "o-",
        label=label,
        color=colors[solver],
        markersize=3,
    )
    final = subset.iloc[-1]
    axis.annotate(
        f"{final['cumulative_seconds']:.2f} s",
        (final["monte_carlo_iteration"], final["cumulative_seconds"]),
        xytext=(5, 0), textcoords="offset points", va="center", fontsize=8,
    )
axis.set_xlabel("Monte Carlo iteration")
axis.set_ylabel("cumulative solve time [s]")
axis.set_title("Cumulative cost across changing matrices")
axis.set_xlim(1, per_sample["monte_carlo_iteration"].max() * 1.05)
axis.grid(alpha=0.25)
axis.legend(frameon=False)
fig.tight_layout(); plt.show()


# 4 · JacobiGMRESLCA in bw2calc

In Brightway, `bw2calc.JacobiGMRESLCA` changes the technosphere solve while leaving biosphere and
characterization processing unchanged. Main arguments are `demand`, `method`, `rtol`, `use_guess`,
`restart`, and `maxiter`.

For repeated solves, `use_guess=True` starts from the previous supply array; `False` starts each
solve without that warm start.

```python
from bw2calc import JacobiGMRESLCA
lca = JacobiGMRESLCA({activity.id: 1.0}, method, rtol=1e-4, use_guess=True)
lca.lci()
lca.lcia()
print(lca.score)
```

Check convergence, residual, and agreement before interpreting an iterative score.


# When to use **Jacobi + GMRES**

**Good fit when:**

- The system is **very large and sparse**.
- The matrix **changes repeatedly**.
- A direct factorization is close to the machine's **memory limit**.
- A **controlled tolerance** is acceptable.

**Choose UMFPACK or Pardiso when:**

- The matrix is small or moderate.
- Many demands reuse one fixed matrix.
- You need a direct answer and factorization fits comfortably in memory.

In [ ]:
# Reproducible live comparison on one randomly selected BAFU activity.
import os
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

import bw2calc as bc
import bw2data as bd

PROJECT = os.environ.get("BRIGHTCON_BW_PROJECT", "clic-bafu-2025-ef31")
DATABASE = "bafu"
METHOD = ("IPCC 2021", "climate change", "global warming potential (GWP100)")
BAFU_SEED = 2026

available_projects = {project.name for project in bd.projects}
if PROJECT not in available_projects:
    display(Markdown(f"**SKIPPED** — Brightway project `{PROJECT}` is not available. Set `BRIGHTCON_BW_PROJECT` to the prepared project name."))
else:
    bd.projects.set_current(PROJECT)
    if DATABASE not in bd.databases:
        display(Markdown(f"**SKIPPED** — database `{DATABASE}` is not present in `{PROJECT}`."))
    elif METHOD not in bd.methods:
        display(Markdown(f"**SKIPPED** — the requested IPCC 2021 method is not present in `{PROJECT}`."))
    else:
        activities = sorted(bd.Database(DATABASE), key=lambda activity: str(activity.key))
        selection_index = int(np.random.default_rng(BAFU_SEED).integers(len(activities)))
        activity = activities[selection_index]
        demand = {activity.id: 1.0}

        started = perf_counter()
        direct = bc.LCA(demand, method=METHOD)
        direct.lci()
        direct.lcia()
        direct_seconds = perf_counter() - started

        started = perf_counter()
        iterative = bc.JacobiGMRESLCA(
            demand, method=METHOD, rtol=RTOL, use_guess=False
        )
        iterative.lci()
        iterative.lcia()
        iterative_seconds = perf_counter() - started

        def lca_residual(lca):
            numerator = np.linalg.norm(
                lca.technosphere_matrix @ lca.supply_array - lca.demand_array
            )
            return float(numerator / np.linalg.norm(lca.demand_array))

        direct_residual = lca_residual(direct)
        iterative_residual = lca_residual(iterative)
        score_difference = abs(iterative.score - direct.score)
        relative_score_difference = score_difference / max(
            abs(direct.score), np.finfo(float).tiny
        )
        method_unit = bd.Method(METHOD).metadata.get("unit", "")
        matrix = direct.technosphere_matrix

        display(Markdown(
            f"**LIVE · seed {BAFU_SEED}** — `{activity['code']}`: "
            f"{activity.get('name', 'unnamed')}<br>"
            f"{activity.get('location', 'unknown location')} · "
            f"1 {activity.get('unit', 'unit')} · matrix {matrix.shape[0]:,} × {matrix.shape[1]:,}, "
            f"{matrix.nnz:,} nonzeros"
        ))
        comparison = pd.DataFrame([
            {"solver": "bw2calc.LCA", "wall_seconds": direct_seconds, "score": direct.score, "relative_residual": direct_residual, "relative_score_difference": 0.0},
            {"solver": "bw2calc.JacobiGMRESLCA", "wall_seconds": iterative_seconds, "score": iterative.score, "relative_residual": iterative_residual, "relative_score_difference": relative_score_difference},
        ])
        display(comparison.style.format({
            "wall_seconds": "{:.3f}",
            "score": "{:.6g}",
            "relative_residual": "{:.2e}",
            "relative_score_difference": "{:.3%}",
        }).hide(axis="index"))
        display(Markdown(
            f"Score unit: **{method_unit or 'method unit not recorded'}**. "
            f"Jacobi + GMRES uses `rtol={RTOL:g}`; its residual and score difference are reported explicitly."
        ))
        assert iterative_residual <= RTOL * 1.01


In [ ]:
# Paired stochastic comparison using the same BAFU activity and LCIA method.
import hashlib
import warnings

display(Markdown("## Paired Monte Carlo comparison"))
MC_ITERATIONS = 500

if "activity" not in globals():
    display(Markdown("**SKIPPED** — the live BAFU calculation above did not run."))
else:
    def sparse_fingerprint(matrix):
        canonical = matrix.tocsr(copy=True)
        canonical.sum_duplicates()
        canonical.eliminate_zeros()
        canonical.sort_indices()
        digest = hashlib.blake2b(digest_size=12)
        for array in (canonical.indptr, canonical.indices, canonical.data):
            digest.update(np.ascontiguousarray(array).view(np.uint8))
        return digest.hexdigest()

    class StochasticJacobiGMRESLCA(bc.JacobiGMRESLCA):
        # bw2calc 2.5.0 needs the prepared matrix rebound after resampling.
        def after_matrix_iteration(self):
            self.technosphere_matrix = self.technosphere_mm.matrix
            self._matrix_prepared = False
            self._cached_preconditioner = None

    def run_monte_carlo(calculator, **solver_options):
        lca = calculator(
            demand=demand,
            method=METHOD,
            use_distributions=True,
            seed_override=BAFU_SEED,
            **solver_options,
        )
        lca.load_lci_data()
        lca.load_lcia_data()
        scores, residuals, sample_seconds = [], [], []
        technosphere_fingerprints, biosphere_fingerprints = [], []

        for sample in range(MC_ITERATIONS):
            started = perf_counter()
            if sample == 0:
                lca.lci()
                lca.lcia()
            else:
                next(lca)
            sample_seconds.append(perf_counter() - started)
            scores.append(float(lca.score))
            residuals.append(lca_residual(lca))
            technosphere_fingerprints.append(
                sparse_fingerprint(lca.technosphere_matrix)
            )
            biosphere_fingerprints.append(sparse_fingerprint(lca.biosphere_matrix))

        return {
            "scores": np.asarray(scores),
            "residuals": np.asarray(residuals),
            "sample_seconds": np.asarray(sample_seconds),
            "technosphere_fingerprints": technosphere_fingerprints,
            "biosphere_fingerprints": biosphere_fingerprints,
        }

    from scikits.umfpack import UmfpackWarning

    with warnings.catch_warnings(record=True) as direct_warnings:
        warnings.simplefilter("always", UmfpackWarning)
        direct_mc = run_monte_carlo(bc.LCA)
    umfpack_warning_count = sum(
        issubclass(item.category, UmfpackWarning) for item in direct_warnings
    )

    import bw2calc.jacobi_gmres_lca as jacobi_module

    gmres_iterations, gmres_info = [], []
    original_gmres = jacobi_module.gmres

    def monitored_gmres(*args, **kwargs):
        residual_history = []
        kwargs["callback"] = residual_history.append
        kwargs["callback_type"] = "pr_norm"
        solution, info = original_gmres(*args, **kwargs)
        gmres_iterations.append(len(residual_history))
        gmres_info.append(int(info))
        return solution, info

    jacobi_module.gmres = monitored_gmres
    try:
        iterative_mc = run_monte_carlo(
            StochasticJacobiGMRESLCA,
            rtol=RTOL,
            use_guess=True,
        )
    finally:
        jacobi_module.gmres = original_gmres

    matrices_match = (
        direct_mc["technosphere_fingerprints"]
        == iterative_mc["technosphere_fingerprints"]
        and direct_mc["biosphere_fingerprints"]
        == iterative_mc["biosphere_fingerprints"]
    )
    absolute_score_difference = np.abs(
        iterative_mc["scores"] - direct_mc["scores"]
    )
    relative_score_difference = absolute_score_difference / np.maximum(
        np.abs(direct_mc["scores"]), np.finfo(float).tiny
    )

    mc_summary = pd.DataFrame([
        {
            "solver": "bw2calc.LCA",
            "total_solve_seconds": direct_mc["sample_seconds"].sum(),
            "median_sample_ms": 1000 * np.median(direct_mc["sample_seconds"]),
            "mean_score": direct_mc["scores"].mean(),
            "score_std": direct_mc["scores"].std(ddof=1),
            "max_relative_residual": direct_mc["residuals"].max(),
            "median_gmres_iterations": np.nan,
        },
        {
            "solver": "bw2calc.JacobiGMRESLCA",
            "total_solve_seconds": iterative_mc["sample_seconds"].sum(),
            "median_sample_ms": 1000 * np.median(iterative_mc["sample_seconds"]),
            "mean_score": iterative_mc["scores"].mean(),
            "score_std": iterative_mc["scores"].std(ddof=1),
            "max_relative_residual": iterative_mc["residuals"].max(),
            "median_gmres_iterations": np.median(gmres_iterations),
        },
    ])
    display(Markdown(
        f"**LIVE · {MC_ITERATIONS} paired samples · seed {BAFU_SEED}** — "
        f"same activity `{activity['code']}` and method. "
        f"All matrix fingerprints match: **{matrices_match}**. "
        f"UMFPACK near-singular warnings: **{umfpack_warning_count}**."
    ))
    display(mc_summary.style.format({
        "total_solve_seconds": "{:.2f}",
        "median_sample_ms": "{:.1f}",
        "mean_score": "{:.6g}",
        "score_std": "{:.3g}",
        "max_relative_residual": "{:.2e}",
        "median_gmres_iterations": "{:.0f}",
    }, na_rep="—").hide(axis="index"))
    display(Markdown(
        f"Median relative score difference: **{np.median(relative_score_difference):.3%}**; "
        f"maximum: **{relative_score_difference.max():.3%}**. "
        "Jacobi uses warm starts and the stochastic matrix-rebinding safeguard required by `bw2calc 2.5.0`."
    ))

    samples = np.arange(1, MC_ITERATIONS + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(samples, np.cumsum(direct_mc["sample_seconds"]), label="bw2calc.LCA", color=colors.get("umfpack"))
    axes[0].plot(samples, np.cumsum(iterative_mc["sample_seconds"]), label="Jacobi + GMRES", color=colors["jacobi-gmres"])
    axes[0].set(xlabel="Monte Carlo sample", ylabel="cumulative solve time [s]", title="Changing matrix: cumulative cost")
    axes[0].legend(frameon=False)

    axes[1].scatter(direct_mc["scores"], iterative_mc["scores"], s=10, alpha=0.45, color=colors["jacobi-gmres"])
    score_limits = [
        min(direct_mc["scores"].min(), iterative_mc["scores"].min()),
        max(direct_mc["scores"].max(), iterative_mc["scores"].max()),
    ]
    axes[1].plot(score_limits, score_limits, "--", color="0.35", linewidth=1)
    axes[1].set(xlabel="bw2calc.LCA score", ylabel="Jacobi + GMRES score", title="Paired LCIA score agreement")
    for axis in axes:
        axis.grid(alpha=0.25)
    fig.tight_layout(); plt.show()

    assert matrices_match
    assert len(gmres_info) == MC_ITERATIONS and not any(gmres_info)
    assert iterative_mc["residuals"].max() <= RTOL * 1.01
